# Language Model Comparison

Compare trained neural checkpoints with n-gram baselines for one configured corpus. Enter the corpus key (for example, webtext). The notebook reads its YAML config, uses the existing processed train/validation/test files, and loads checkpoints from the corresponding results folder.

This notebook does not run the data pipeline or train neural models. It builds n-gram count tables from the training split and evaluates all selected models on the same validation and test splits. Run it from the repository checkout, or set PROJECT_ROOT to that checkout in the first code cell.


## 1. Runtime and project setup

The checkout must contain src/lm, src/configs, processed split files, and trained results.

In [17]:
from pathlib import Path
import os, sys, json, time, re
import yaml
import pandas as pd
import torch
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src/lm/cli.py").is_file() and (candidate / "src/configs").is_dir():
            return candidate
    return None

PROJECT_ROOT = None  # Set this to Path('/content/your-checkout') in Colab if auto-detection fails.
if PROJECT_ROOT is None:
    PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the repository checkout path.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project:", PROJECT_ROOT)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)


Project: /Users/muralidm/study/NLP/assignment/code/bits-nlp-assignment
Python: 3.13.5 | PyTorch: 2.12.0


## 2. Choose corpus and load its config

Enter the stem of src/configs/<corpus>.yaml. Paths for the processed splits and results are inferred from the config.

In [ ]:
CORPUS = "roc-stories"
TRAIN_PERCENT = 100  # Select the matching train_<percent>pct result folder.
if not CORPUS:
    raise ValueError("Enter a corpus name.")
CONFIG_PATH = PROJECT_ROOT / "src/configs" / f"{CORPUS}.yaml"
if not CONFIG_PATH.is_file():
    available = sorted(p.stem for p in (PROJECT_ROOT / "src/configs").glob("*.yaml"))
    raise FileNotFoundError(f"No config {CONFIG_PATH.name}. Available: {available}")

CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_CONFIG = CONFIG.get("data", {})
if DATA_CONFIG.get("corpus") != CORPUS:
    raise ValueError(f"Config declares data.corpus={DATA_CONFIG.get('corpus')!r}, not {CORPUS!r}.")

def project_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()

CORPUS_DIR = project_path(DATA_CONFIG.get("output", "data/processed")) / CORPUS
SOURCE_TRAIN_PATH = CORPUS_DIR / "train.txt"
VALID_PATH, TEST_PATH = (CORPUS_DIR / name for name in ("valid.txt", "test.txt"))
RARE_CSVS = {
    "Unseen bigram sequences": CORPUS_DIR / "rare-seq-bigram.csv",
    "Unseen trigram sequences": CORPUS_DIR / "rare-seq-trigram.csv",
}
RESULT_BASE = project_path(CONFIG.get("output_dir", "results")) / CONFIG["name"]
RESULT_ROOT = RESULT_BASE / f"train_{TRAIN_PERCENT:g}pct"
# Fall back to output folders made by earlier runner versions.
if not RESULT_ROOT.is_dir() and RESULT_BASE.is_dir():
    RESULT_ROOT = RESULT_BASE
TRAIN_SNAPSHOT_PATH = RESULT_ROOT / "train.txt"
TRAIN_PATH = TRAIN_SNAPSHOT_PATH if TRAIN_SNAPSHOT_PATH.is_file() else SOURCE_TRAIN_PATH
TRAIN_FILE_PERCENT = 100 if TRAIN_SNAPSHOT_PATH.is_file() else TRAIN_PERCENT
for path in (TRAIN_PATH, VALID_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Prepare the corpus separately first.")
print("Config:", CONFIG_PATH.relative_to(PROJECT_ROOT))
print("Data:", CORPUS_DIR.relative_to(PROJECT_ROOT))
print("Results:", RESULT_ROOT.relative_to(PROJECT_ROOT))
print("Configured min_freq:", CONFIG.get("vocab", {}).get("min_freq", 2))


Config: src/configs/roc-stories.yaml
Data: data/processed/roc-stories
Results: results/roc-stories
Configured min_freq: 1


## 3. Select and load trained neural checkpoints

Press Enter to load every checkpoint found under the configured results folder, or enter selected numbers separated by commas. Checkpoints need their adjacent config.yaml and vocab.json.

In [19]:
print(RESULT_ROOT)

/Users/muralidm/study/NLP/assignment/code/bits-nlp-assignment/results/roc-stories


In [20]:
from lm.cli import LanguageModelCLI
CHECKPOINTS = sorted(RESULT_ROOT.rglob("*_seed*.pt"))
if not CHECKPOINTS:
    raise FileNotFoundError(f"No neural checkpoints found under {RESULT_ROOT}")
for i, path in enumerate(CHECKPOINTS, 1):
    print(f"{i}. {path.relative_to(PROJECT_ROOT)}")
choice = "1,2,3"
if choice != "":
    indices = [int(x.strip()) for x in choice.split(",")]
    if any(i < 1 or i > len(CHECKPOINTS) for i in indices):
        raise ValueError("Invalid checkpoint number.")
    SELECTED = [CHECKPOINTS[i - 1] for i in indices]
else:
    SELECTED = CHECKPOINTS

CLI = LanguageModelCLI(CONFIG.get("device", "auto"))
NEURAL_MODELS = []
for path in SELECTED:
    key = CLI.load_neural(path)
    info = CLI.neural[key]
    info["comparison_label"] = info["name"]
    if info["name"] == "ffnn":
        run_config = yaml.safe_load((path.parent / "config.yaml").read_text(encoding="utf-8"))
        params = run_config.get("models", [{}])[0].get("params", {})
        context_size = params.get("context_size")
        if context_size is not None:
            info["comparison_label"] = f"ffnn_ctx{context_size}"
    NEURAL_MODELS.append(info)

if len({tuple(x["vocab"]) for x in NEURAL_MODELS}) != 1:
    raise ValueError("Selected checkpoints have different vocabularies; select a matching run.")
if (len({x["lowercase"] for x in NEURAL_MODELS}) != 1
        or len({x["train_percent"] for x in NEURAL_MODELS}) != 1
        or len({x["tokenizer"] for x in NEURAL_MODELS}) != 1):
    raise ValueError("Selected checkpoints use different preprocessing, tokenizers, or training percentages.")
for info in NEURAL_MODELS:
    if info["train_path"].resolve() != TRAIN_PATH.resolve():
        raise ValueError(f"{info['checkpoint'].name} used a different training file: {info['train_path']}")
print(f"Loaded {len(NEURAL_MODELS)} checkpoints on {CLI.device}.")


1. results/roc-stories/models/ffnn_ctx1/ffnn_seed13.pt
2. results/roc-stories/models/ffnn_ctx2/ffnn_seed13.pt
3. results/roc-stories/models/lstm/lstm_seed13.pt
Loaded neural ffnn: ffnn_seed13.pt (vocab 59,440, device mps)
Loaded neural ffnn: ffnn_seed13.pt (vocab 59,440, device mps)
Loaded neural lstm: lstm_seed13.pt (vocab 59,440, device mps)
Loaded 3 checkpoints on mps.


## 4. Audit splits, vocabulary, and OOV rates

Use the vocabulary saved with the neural run for each n-gram model. The split OOV rates below are computed against that training-derived vocabulary.

In [21]:
from lm.data import Vocab
from lm.cli import iter_sentences
VOCAB = Vocab.from_list(NEURAL_MODELS[0]["vocab"])
LOWERCASE = NEURAL_MODELS[0]["lowercase"]
CHECKPOINT_TRAIN_PERCENT = NEURAL_MODELS[0]["train_percent"]
if CHECKPOINT_TRAIN_PERCENT != TRAIN_PERCENT:
    raise ValueError(f"Selected checkpoint uses {CHECKPOINT_TRAIN_PERCENT:g}% training data, not requested {TRAIN_PERCENT:g}%.")
TOKENIZER = NEURAL_MODELS[0]["tokenizer"]

def split_stats(path):
    n_sentences = n_tokens = n_oov = 0
    for words in iter_sentences(path, LOWERCASE, tokenizer_name=TOKENIZER):
        n_sentences += 1
        n_tokens += len(words)
        n_oov += sum(word not in VOCAB.stoi for word in words)
    return {"sentences": n_sentences, "word_tokens": n_tokens, "oov_tokens": n_oov,
            "oov_rate": n_oov / n_tokens if n_tokens else 0.0}

AUDIT = pd.DataFrame([
    {"split": name, **split_stats(path)}
    for name, path in (("train", TRAIN_PATH), ("valid", VALID_PATH), ("test", TEST_PATH))
])
display(AUDIT.style.format({"oov_rate": "{:.2%}"}))
print(f"Vocabulary size: {len(VOCAB):,}; tokenizer: {TOKENIZER}; n-gram train percentage: {TRAIN_PERCENT:g}%")


,split,sentences,word_tokens,oov_tokens,oov_rate
0,train,356782,3122277,0,0.00%
1,valid,91813,804514,6893,0.86%
2,test,40069,350471,2963,0.85%


Vocabulary size: 59,440; n-gram train percentage: 100%


## 5. Build n-gram baselines

The default matrix includes unsmoothed bigram and trigram, add-k trigram, and discounted backoff trigram. Edit NGRAM_SPECS to match the planned experiment. All models use the same vocabulary and the neural run's training percentage.

In [22]:
from lm.cli import CountNGram
NGRAM_SPECS = [
    {"name":"bigram_mle", "order":2, "smoothing":"mle"},
    {"name":"trigram_mle", "order":3, "smoothing":"mle"},
    {"name":"trigram_laplace_k1", "order":3, "smoothing":"laplace", "k":1.0},
    {"name":"trigram_backoff_d075", "order":3, "smoothing":"backoff", "discount":0.75},
]
NGRAM_MODELS = {}
for spec in NGRAM_SPECS:
    print("Building", spec["name"], "from", TRAIN_PATH.relative_to(PROJECT_ROOT), flush=True)
    started = time.perf_counter()
    model = CountNGram(
        order=spec["order"], vocab=VOCAB.itos,
        sentences=iter_sentences(TRAIN_PATH, LOWERCASE, TRAIN_FILE_PERCENT, TOKENIZER),
        smoothing=spec["smoothing"], k=spec.get("k", 1.0),
        discount=spec.get("discount", 0.75),
        allowed_target_ids=VOCAB.allowed_target_ids,
    )
    elapsed = time.perf_counter() - started
    NGRAM_MODELS[spec["name"]] = {"model":model, "spec":spec, "train_seconds":elapsed,
                                  "train_percent":TRAIN_PERCENT, "tokenizer":TOKENIZER,
                                  "lowercase":LOWERCASE, "train_path":TRAIN_PATH.resolve()}
    print(f"  sentences={model.sentence_count:,}; tokens={model.token_count:,}; "
          f"distinct {spec['order']}-grams={len(model.counts):,}; fit={elapsed:.1f}s")


Building bigram_mle from data/processed/roc-stories/train.txt
  encoded training sentences: 100,000
  encoded training sentences: 200,000
  encoded training sentences: 300,000
  sentences=356,782; tokens=3,122,277; distinct 2-grams=603,014; fit=6.5s
Building trigram_mle from data/processed/roc-stories/train.txt
  encoded training sentences: 100,000
  encoded training sentences: 200,000
  encoded training sentences: 300,000
  sentences=356,782; tokens=3,122,277; distinct 3-grams=1,509,974; fit=25.4s
Building trigram_laplace_k1 from data/processed/roc-stories/train.txt
  encoded training sentences: 100,000
  encoded training sentences: 200,000
  encoded training sentences: 300,000
  sentences=356,782; tokens=3,122,277; distinct 3-grams=1,509,974; fit=14.1s
Building trigram_backoff_d075 from data/processed/roc-stories/train.txt
  encoded training sentences: 100,000
  encoded training sentences: 200,000
  encoded training sentences: 300,000
  sentences=356,782; tokens=3,122,277; distinct 3

## 6. Evaluate on the same validation and test splits

Perplexity includes word targets and one sentence-ending target per sentence. Validation is for diagnostics/model selection; use test for the final reported comparison.

In [16]:
from lm.evaluation import evaluate_model

def evaluate(model, family, path, label, allowed="vocab"):
    started = time.perf_counter()
    result = evaluate_model(
        model=model, vocab=VOCAB, sentences=iter_sentences(path, LOWERCASE, tokenizer_name=TOKENIZER),
        family=family, device=CLI.device if family == "neural" else None,
        allowed_target_ids=allowed, case_limit=25,
    )
    result["evaluation_seconds"] = time.perf_counter() - started
    return result

EVALUATIONS, rows, case_rows = {}, [], []
splits = (("valid", VALID_PATH), ("test", TEST_PATH))
for split_name, split_path in splits:
    for label, item in NGRAM_MODELS.items():
        result = evaluate(item["model"], "ngram", split_path, label)
        EVALUATIONS[(split_name, label)] = result
        rows.append({
            "split":split_name, "model":label, "family":"n-gram",
            "accuracy":result["accuracy"], "perplexity":result["perplexity"],
            "status":result["status"], "correct":result["correct"], "events":result["tokens"],
            "zero_probability_events":result["zero_probability_events"],
            "context_coverage":result["context_covered"] / result["tokens"] if result["tokens"] else None,
            "oov_rate":result["oov_rate"], "training_seconds":item["train_seconds"],
            "evaluation_seconds":result["evaluation_seconds"],
        })
        for case in result["cases"]:
            case_rows.append({"split":split_name, "model":label,
                "prefix":" ".join(case["full_prefix"]), "expected":case["mapped_target"],
                "predicted":case["prediction"], "log_probability":case["log_probability"]})

    for info in NEURAL_MODELS:
        label = info["comparison_label"]
        result = evaluate(info["model"], "neural", split_path, label,
                          info.get("allowed_target_ids", "vocab"))
        EVALUATIONS[(split_name, label)] = result
        train_seconds = None
        runs_path = RESULT_ROOT / "runs.csv"
        seed_match = re.search(r"_seed(-?\d+)$", info["checkpoint"].stem)
        if runs_path.is_file() and seed_match:
            run_table = pd.read_csv(runs_path)
            seed = int(seed_match.group(1))
            variant = info["run_dir"].name
            if {"variant", "seed", "train_seconds"}.issubset(run_table.columns):
                matching = run_table[(run_table["variant"] == variant) & (run_table["seed"] == seed)]
                if not matching.empty:
                    train_seconds = float(matching.iloc[0]["train_seconds"])

        rows.append({
            "split":split_name, "model":label, "family":"neural",
            "accuracy":result["accuracy"], "perplexity":result["perplexity"],
            "status":result["status"], "correct":result["correct"], "events":result["tokens"],
            "zero_probability_events":result["zero_probability_events"],
            "context_coverage":None, "oov_rate":result["oov_rate"],
            "training_seconds":train_seconds, "evaluation_seconds":result["evaluation_seconds"],
        })
        for case in result["cases"]:
            case_rows.append({"split":split_name, "model":label,
                "prefix":" ".join(case["full_prefix"]), "expected":case["mapped_target"],
                "predicted":case["prediction"], "log_probability":case["log_probability"]})

RESULTS = pd.DataFrame(rows)
CASES = pd.DataFrame(case_rows)
display(RESULTS[RESULTS["split"] == "test"].sort_values("perplexity", na_position="last"))


,split,model,family,accuracy,perplexity,status,correct,events,zero_probability_events,context_coverage,oov_rate,training_seconds,evaluation_seconds
8,test,trigram_backoff_d075,n-gram,0.270856,107.774817,finite,105780,390540,0,0.884813,0.008454,14.772481,58.245715
9,test,ffnn_ctx1,neural,0.240057,136.828466,finite,93752,390540,0,NaN,0.008454,2323.097769,203.765083
7,test,trigram_laplace_k1,n-gram,0.234857,6343.206166,finite,91721,390540,0,0.884813,0.008454,16.411428,23.518217
5,test,bigram_mle,n-gram,0.242462,NaN,infinite_zero_probability,94691,390540,46010,0.992413,0.008454,7.488715,1.494020
6,test,trigram_mle,n-gram,0.233587,NaN,infinite_zero_probability,91225,390540,136839,0.884813,0.008454,14.040389,5.785533


## 7. Inspect examples and rare n-gram candidates

The examples shown are the evaluator's first 25 events per model, not a curated sample. Use the rare CSV to select and document at least ten informative cases for the paper; explain the selection rule and include both successes and failures.

In [ ]:
display(CASES[CASES["split"] == "test"].head(50))
for label, rare_csv in RARE_CSVS.items():
    if rare_csv.is_file():
        print(f"{label}: {rare_csv.relative_to(PROJECT_ROOT)}")
        display(pd.read_csv(rare_csv).head(25))
    else:
        print(f"Missing {rare_csv.name}. Run the data pipeline rare-check separately if needed.")


## 8. Save comparison artifacts

The CSVs go beside the trained results. Neural training time is not reconstructed here; it remains available in the controller's run summary when that file contains the matching run.

In [9]:
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
metrics_path = RESULT_ROOT / f"{CORPUS}_comparison_metrics.csv"
cases_path = RESULT_ROOT / f"{CORPUS}_comparison_examples.csv"
RESULTS.to_csv(metrics_path, index=False)
CASES.to_csv(cases_path, index=False)
print("Metrics:", metrics_path.relative_to(PROJECT_ROOT))
print("Examples:", cases_path.relative_to(PROJECT_ROOT))
print("Checkpoints:")
for info in NEURAL_MODELS:
    print(" -", info["checkpoint"].relative_to(PROJECT_ROOT))


Metrics: results/webtext-nn/webtext_comparison_metrics.csv
Examples: results/webtext-nn/webtext_comparison_examples.csv
Checkpoints:
 - results/webtext-nn/models/ffnn_ctx1/ffnn_seed13.pt


## 9. Incremental evaluation cells

This section is separate from the batch evaluation above. Run one model cell to evaluate only that model on validation and test. Its rows update `INCREMENTAL_RESULTS` for that model; results for other models remain untouched. Re-running a cell replaces only its own model entries. The final cell combines whichever model results have been collected so far.


In [ ]:
# Incremental results are keyed by (split, model label). This map is independent of EVALUATIONS/RESULTS above.
if "INCREMENTAL_RESULTS" not in globals():
    INCREMENTAL_RESULTS = {}
if "INCREMENTAL_NGRAM_MODELS" not in globals():
    INCREMENTAL_NGRAM_MODELS = {}
INCREMENTAL_SPLITS = {"valid": VALID_PATH, "test": TEST_PATH}
INCREMENTAL_SEED = None  # Set an integer if the results folder contains multiple seeds per model.

from lm.evaluation import evaluate_model as evaluate_one_model

def _incremental_record(split, label, family, result, train_seconds=None):
    words = result["word_tokens"]
    word_metrics = result["word_metrics"]
    return {
        "split": split,
        "model": label,
        "family": "n-gram" if family == "ngram" else family,
        "accuracy": word_metrics["accuracy"],
        "correct_word_predictions": word_metrics["correct"],
        "word_tokens": words,
        "perplexity": result["perplexity"],
        "status": result["status"],
        "perplexity_events": result["tokens"],
        "sentence_end_targets": result["sentence_end_targets"],
        "zero_probability_events": result["zero_probability_events"],
        "context_coverage": (
            word_metrics["context_covered"] / words if family == "ngram" and words else None
        ),
        "oov_rate": result["oov_rate"],
        "train_seconds": train_seconds,
        "evaluation_seconds": result["evaluation_seconds"],
    }

def _format_incremental_table(frame):
    formatted = frame.copy()
    formats = {
        "accuracy": "{:.2%}", "perplexity": "{:.5g}",
        "context_coverage": "{:.2%}", "oov_rate": "{:.2%}",
        "train_seconds": "{:.1f}", "evaluation_seconds": "{:.1f}",
    }
    for column, spec in formats.items():
        if column in formatted:
            formatted[column] = formatted[column].map(
                lambda value: spec.format(value) if pd.notna(value) else "—"
            )
    return formatted

def _evaluate_incremental(label, model, family, train_seconds=None, allowed="vocab"):
    # Build results locally, then replace only this model's two split entries.
    updated = {}
    for split_name, split_path in INCREMENTAL_SPLITS.items():
        started = time.perf_counter()
        result = evaluate_one_model(
            model=model,
            vocab=VOCAB,
            sentences=iter_sentences(split_path, LOWERCASE, tokenizer_name=TOKENIZER),
            family=family,
            device=CLI.device if family == "neural" else None,
            allowed_target_ids=allowed,
            case_limit=0,
        )
        result["evaluation_seconds"] = time.perf_counter() - started
        updated[(split_name, label)] = _incremental_record(
            split_name, label, family, result, train_seconds
        )
    INCREMENTAL_RESULTS.update(updated)
    display(_format_incremental_table(pd.DataFrame(updated.values())))
    print(f"Updated only {label}; other incremental results were retained.")

def _incremental_ngram(label, order, smoothing, k=1.0, discount=0.75):
    spec = {"order": order, "smoothing": smoothing, "k": k, "discount": discount,
            "train_percent": TRAIN_PERCENT, "tokenizer": TOKENIZER,
            "lowercase": LOWERCASE, "train_path": str(TRAIN_PATH.resolve())}
    cached = INCREMENTAL_NGRAM_MODELS.get(label)
    if cached is None or cached["spec"] != spec:
        # Reuse an equivalent model from the original batch section when available.
        batch_models = globals().get("NGRAM_MODELS", {})
        reusable = next((item for item in batch_models.values()
            if item["spec"].get("order") == order
            and item["spec"].get("smoothing") == smoothing
            and (smoothing != "laplace" or float(item["spec"].get("k", 1.0)) == float(k))
            and (smoothing != "backoff" or float(item["spec"].get("discount", 0.75)) == float(discount))
            and item.get("train_percent") == TRAIN_PERCENT
            and item.get("tokenizer") == TOKENIZER
            and item.get("lowercase") == LOWERCASE
            and item.get("train_path") == TRAIN_PATH.resolve()), None)
        if reusable is not None:
            model = reusable["model"]
            train_seconds = reusable["train_seconds"]
            print(f"Reusing existing batch counts for {label}; evaluating only this model.")
        else:
            print(f"Building {label} from {TRAIN_PATH.relative_to(PROJECT_ROOT)}", flush=True)
            started = time.perf_counter()
            model = CountNGram(
                order=order,
                vocab=VOCAB.itos,
                sentences=iter_sentences(TRAIN_PATH, LOWERCASE, TRAIN_FILE_PERCENT, TOKENIZER),
                smoothing=smoothing,
                k=k,
                discount=discount,
                allowed_target_ids=VOCAB.allowed_target_ids,
            )
            train_seconds = time.perf_counter() - started
            print(f"Built {label}: {model.sentence_count:,} sentences; {len(model.counts):,} distinct {order}-grams; {train_seconds:.1f}s")
        INCREMENTAL_NGRAM_MODELS[label] = {
            "model": model, "spec": spec, "train_seconds": train_seconds,
        }
    else:
        model = cached["model"]
        train_seconds = cached["train_seconds"]
        print(f"Reusing cached counts for {label}; evaluating only this model.")
    _evaluate_incremental(label, model, "ngram", train_seconds)

def _incremental_neural(label):
    # Prefer a checkpoint already loaded above; otherwise load the matching model folder on demand.
    matches = [info for info in NEURAL_MODELS if info.get("comparison_label") == label]
    if matches:
        info = matches[0]
    else:
        paths = [path for path in CHECKPOINTS if path.parent.name == label]
        if INCREMENTAL_SEED is not None:
            paths = [path for path in paths if f"_seed{INCREMENTAL_SEED}" in path.stem]
        elif len(paths) > 1:
            reference = NEURAL_MODELS[0]["checkpoint"].stem if NEURAL_MODELS else ""
            seed = re.search(r"_seed(-?\d+)$", reference)
            if seed:
                paths = [path for path in paths if f"_seed{seed.group(1)}" in path.stem]
        if len(paths) != 1:
            raise ValueError(
                f"Need exactly one checkpoint for {label}; found {len(paths)}. "
                "Set INCREMENTAL_SEED to select a seed."
            )
        key = CLI.load_neural(paths[0])
        info = CLI.neural[key]
        info["comparison_label"] = label
    if tuple(info["vocab"]) != tuple(VOCAB.itos):
        raise ValueError(f"{label} uses a different vocabulary from the selected experiment.")
    if (info["lowercase"] != LOWERCASE or info["train_percent"] != TRAIN_PERCENT
            or info["tokenizer"] != TOKENIZER):
        raise ValueError(f"{label} uses different preprocessing, tokenizer, or training percentage.")
    if info["train_path"].resolve() != TRAIN_PATH.resolve():
        raise ValueError(f"{label} was trained from a different train.txt.")
    print(f"Evaluating {label}: {info['checkpoint'].relative_to(PROJECT_ROOT)}")
    _evaluate_incremental(
        label, info["model"], "neural", allowed=info.get("allowed_target_ids", "vocab")
    )


### Neural models

Each cell evaluates one neural checkpoint. If multiple seeds exist, set `INCREMENTAL_SEED` in the setup cell to select the same seed across models.


In [ ]:
_incremental_neural("ffnn_ctx1")

In [ ]:
_incremental_neural("ffnn_ctx2")

In [ ]:
_incremental_neural("lstm")

### N-gram models

Each cell builds or reuses that model’s counts, then evaluates only that model on validation and test. Laplace uses `k=1`; backoff uses discount `0.75`. Change those arguments in the specific cell to explore other settings.


In [ ]:
_incremental_ngram("bigram_mle", 2, "mle")

In [ ]:
_incremental_ngram("bigram_laplace", 2, "laplace", k=1.0)

In [ ]:
_incremental_ngram("bigram_backoff", 2, "backoff", discount=0.75)

In [ ]:
_incremental_ngram("trigram_mle", 3, "mle")

In [ ]:
_incremental_ngram("trigram_laplace", 3, "laplace", k=1.0)

In [ ]:
_incremental_ngram("trigram_backoff", 3, "backoff", discount=0.75)

### Combine the incremental results

Run this cell after any subset of the model cells. It combines only entries currently in `INCREMENTAL_RESULTS`; optionally set `SAVE_INCREMENTAL_TABLE = True` to write the combined metrics CSV.


In [ ]:
INCREMENTAL_TABLE = pd.DataFrame(INCREMENTAL_RESULTS.values())
if INCREMENTAL_TABLE.empty:
    print("Run one or more incremental model cells first.")
else:
    INCREMENTAL_TABLE = INCREMENTAL_TABLE.sort_values(["split", "model"]).reset_index(drop=True)
    display(_format_incremental_table(INCREMENTAL_TABLE))
    SAVE_INCREMENTAL_TABLE = False
    if SAVE_INCREMENTAL_TABLE:
        incremental_path = RESULT_ROOT / f"{CORPUS}_incremental_metrics.csv"
        INCREMENTAL_TABLE.to_csv(incremental_path, index=False)
        print("Saved:", incremental_path.relative_to(PROJECT_ROOT))


## 10. Overall results: bigram and trigram comparisons

The plots below use whichever evaluation results are available from the batch evaluation and incremental model cells. Bigram models are shown with `ffnn_ctx1` and `lstm`; trigram models are shown with `ffnn_ctx2` and `lstm`. Validation and test are plotted separately. Re-running an incremental evaluation updates that model's plotted values without replacing the others. Perplexity includes sentence-ending targets; accuracy is over word targets.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Merge batch and incremental results. Incremental rows take precedence for a
# model/split pair, so a freshly rerun model appears without stale duplicates.
_plot_frames = []
if "RESULTS" in globals() and isinstance(RESULTS, pd.DataFrame) and not RESULTS.empty:
    _plot_frames.append(RESULTS.copy())
if "INCREMENTAL_RESULTS" in globals() and INCREMENTAL_RESULTS:
    _plot_frames.append(pd.DataFrame(INCREMENTAL_RESULTS.values()))

if not _plot_frames:
    raise RuntimeError("No evaluation results are available. Run the evaluation cells first.")

PLOT_RESULTS = pd.concat(_plot_frames, ignore_index=True, sort=False)
PLOT_RESULTS = PLOT_RESULTS.drop_duplicates(["split", "model"], keep="last")
PLOT_RESULTS["accuracy"] = pd.to_numeric(PLOT_RESULTS["accuracy"], errors="coerce")
PLOT_RESULTS["perplexity"] = pd.to_numeric(PLOT_RESULTS["perplexity"], errors="coerce")

# Infer n-gram order from the model specifications, with label prefixes as a
# fallback for result rows loaded from a prior batch run.
NGRAM_ORDER_BY_LABEL = {}
for _collection_name in ("NGRAM_MODELS", "INCREMENTAL_NGRAM_MODELS"):
    for _label, _item in globals().get(_collection_name, {}).items():
        _spec = _item.get("spec", {})
        if _spec.get("order") in (2, 3):
            NGRAM_ORDER_BY_LABEL[_label] = int(_spec["order"])
for _label in PLOT_RESULTS["model"].dropna().astype(str).unique():
    _normalized = _label.lower().replace("-", "_")
    if _normalized.startswith(("bigram", "bi_")):
        NGRAM_ORDER_BY_LABEL.setdefault(_label, 2)
    elif _normalized.startswith(("trigram", "tri_")):
        NGRAM_ORDER_BY_LABEL.setdefault(_label, 3)

# A compact boxed table reports split sizes and OOV rates against the shared
# train-derived vocabulary, plus the experiment-level vocabulary settings.
if "AUDIT" not in globals() or AUDIT.empty:
    raise RuntimeError("Split audit is unavailable. Run the vocabulary/OOV audit cell first.")
_summary_rows = []
for _, _row in AUDIT.iterrows():
    _summary_rows.append([
        str(_row["split"]),
        f"{int(_row['sentences']):,}",
        f"{int(_row['word_tokens']):,}",
        f"{int(_row['oov_tokens']):,}",
        f"{float(_row['oov_rate']):.2%}",
    ])
fig, ax = plt.subplots(figsize=(10, max(1.8, 0.48 * len(_summary_rows) + 1.2)))
ax.axis("off")
_summary_table = ax.table(
    cellText=_summary_rows,
    colLabels=["Split", "Sentences", "Word tokens", "OOV tokens", "OOV rate"],
    cellLoc="center", loc="center",
)
_summary_table.auto_set_font_size(False)
_summary_table.set_fontsize(10)
_summary_table.scale(1, 1.45)
for _cell in _summary_table.get_celld().values():
    _cell.set_edgecolor("#555555")
    _cell.set_linewidth(0.8)
    _cell.set_facecolor("#f3f6fa" if _cell.get_text().get_text() else "white")
for _column in range(5):
    _summary_table[(0, _column)].set_facecolor("#dbe7f3")
    _summary_table[(0, _column)].get_text().set_weight("bold")
ax.set_title(
    f"{CORPUS} evaluation data  |  vocabulary: {len(VOCAB):,}  |  train portion: {TRAIN_PERCENT:g}%",
    fontsize=12, pad=14,
)
plt.tight_layout()
plt.show()


def plot_order_comparison(order, neural_labels, title):
    _subset = PLOT_RESULTS[PLOT_RESULTS["model"].map(
        lambda _label: NGRAM_ORDER_BY_LABEL.get(str(_label)) == order
        or str(_label).lower() in neural_labels
    )].copy()
    _subset = _subset[_subset["split"].isin(["valid", "test"])]
    if _subset.empty:
        print(f"No {title.lower()} evaluation results found.")
        return

    _models = list(dict.fromkeys(_subset["model"].astype(str)))
    _splits = [name for name in ("valid", "test") if name in set(_subset["split"])]
    _x = np.arange(len(_models))
    _width = 0.8 / max(1, len(_splits))
    _colors = {"valid": "#8ab6d6", "test": "#e39a63"}
    _fig, (_accuracy_ax, _ppl_ax) = plt.subplots(1, 2, figsize=(max(12, len(_models) * 1.7), 5))
    _infinity_marker_top = 1.0

    for _split_index, _split in enumerate(_splits):
        _rows = _subset[_subset["split"] == _split].set_index("model")
        _accuracies = [
            float(_rows.loc[_model, "accuracy"]) * 100
            if _model in _rows.index and pd.notna(_rows.loc[_model, "accuracy"]) else 0
            for _model in _models
        ]
        _positions = _x + (_split_index - (len(_splits) - 1) / 2) * _width
        _bars = _accuracy_ax.bar(
            _positions, _accuracies, _width, label=_split,
            color=_colors.get(_split, "#999999"), edgecolor="white",
        )
        _accuracy_ax.bar_label(_bars, fmt="%.1f%%", padding=2, fontsize=8, rotation=0)

        _finite_values = []
        _infinite_positions = []
        for _position, _model in zip(_positions, _models):
            _value = _rows.loc[_model, "perplexity"] if _model in _rows.index else np.nan
            if pd.notna(_value) and np.isfinite(float(_value)) and float(_value) > 0:
                _finite_values.append((float(_position), float(_value)))
            elif (_model in _rows.index and pd.notna(_value)
                  and np.isinf(float(_value))):
                _infinite_positions.append(float(_position))
        if _finite_values:
            _bars = _ppl_ax.bar(
                [item[0] for item in _finite_values],
                [item[1] for item in _finite_values],
                _width, label=_split, color=_colors.get(_split, "#999999"), edgecolor="white",
            )
        else:
            _bars = []
        _split_finite_max = max((item[1] for item in _finite_values), default=1.0)
        _infinity_marker_top = max(_infinity_marker_top, _split_finite_max * 1.5)
        for _position in _infinite_positions:
            _ppl_ax.scatter([_position], [_infinity_marker_top], marker="^", s=55,
                            color=_colors.get(_split, "#999999"), zorder=4,
                            label=_split if not _finite_values and _position == _infinite_positions[0] else None)
            _ppl_ax.annotate("∞", (_position, _infinity_marker_top), xytext=(0, 6),
                             textcoords="offset points", ha="center", fontsize=10)

    _accuracy_ax.set_title("Top-1 accuracy")
    _accuracy_ax.set_ylabel("Correct word targets (%)")
    _max_accuracy = _subset["accuracy"].max(skipna=True)
    _accuracy_ax.set_ylim(0, min(100, max(5, float(_max_accuracy) * 120 if pd.notna(_max_accuracy) else 5)))
    _accuracy_ax.set_xticks(_x, _models, rotation=30, ha="right")
    _accuracy_ax.grid(axis="y", alpha=0.25)
    _accuracy_ax.legend(title="Split")

    _ppl_ax.set_title("Perplexity")
    _ppl_ax.set_ylabel("Perplexity (log scale; ∞ marks zero-probability cases)")
    _ppl_ax.set_yscale("log")
    _ppl_ax.set_ylim(bottom=1, top=_infinity_marker_top * 2)
    _ppl_ax.set_xticks(_x, _models, rotation=30, ha="right")
    _ppl_ax.grid(axis="y", which="both", alpha=0.25)
    _ppl_ax.legend(title="Split")

    _fig.suptitle(title, fontsize=14)
    _fig.tight_layout()
    plt.show()
    display(_subset[["split", "model", "accuracy", "perplexity", "zero_probability_events"]
                    if "zero_probability_events" in _subset.columns
                    else ["split", "model", "accuracy", "perplexity"]]
            .sort_values(["model", "split"]).reset_index(drop=True))


plot_order_comparison(2, {"ffnn_ctx1", "lstm"}, "Bigram models: n-gram variants, FFNN context 1, and LSTM")
plot_order_comparison(3, {"ffnn_ctx2", "lstm"}, "Trigram models: n-gram variants, FFNN context 2, and LSTM")
